In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

# Energy Signature

```{admonition} Goal
:class: tip
Plot an energy signature showing heating demand as a function of outdoor temperature. This reveals the building's heating behavior and the balance point temperature.
```

```{admonition} Data Basis
:class: note
Two datasets are used:
- `centralHeating.csv` — cumulative heating energy and supply temperature (columns: `time`, `energyHeatingMeter`, `supplyTempHeating`; separator `;`)
- `centralOutsideTemp.csv` — outdoor temperature (columns: `time`, `centralOutsideTemp`; separator `;`)

Both are merged on time into a 3-column DataFrame [timestamp, outside_temp, heating_value].
```

## Data Preparation

In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/"

# --- Heating data ---
raw_heat = pd.read_csv(base_url + "centralHeating.csv", sep=";")
raw_heat["time"] = pd.to_datetime(raw_heat["time"])
raw_heat = raw_heat.set_index("time")

# Compute heating consumption from cumulative meter
raw_heat["heating"] = raw_heat["energyHeatingMeter"].diff()

# --- Outdoor temperature ---
raw_out = pd.read_csv(base_url + "centralOutsideTemp.csv", sep=";")
raw_out["time"] = pd.to_datetime(raw_out["time"])
raw_out = raw_out.set_index("time")

# Merge on time index
merged = raw_out.join(raw_heat[["heating"]], how="inner")

# Build 3-column DataFrame [timestamp, outside_temp, heating_value]
df = merged.reset_index()
df = df[["time", "centralOutsideTemp", "heating"]].copy()
df.columns = ["timestamp", "outside_temp", "heating_value"]
df = df.dropna()

df.head()

## Visualization

In [ ]:
from pyedautils.plots import plot_energy_signature

fig = plot_energy_signature(df, title="Energy Signature — Heating", xlab="Outside Temperature [°C]", ylab="Heating Power [kW]")
fig.show()

```{dropdown} Customization
You can adjust the axis labels and title. To analyze a different time period, filter the DataFrame before plotting.

```python
df_winter = df[df["timestamp"].dt.month.isin([11, 12, 1, 2, 3])]
fig = plot_energy_signature(df_winter, title="Energy Signature — Winter Only")
```
```